# Trader Performance vs Market Sentiment
### Primetrade.ai — Round 0 Assignment

---

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams['font.size'] = 11
os.makedirs('charts', exist_ok=True)

## Part A — Data Preparation
### A1. Load Datasets

In [ ]:
trades    = pd.read_csv("historical_data.csv")
sentiment = pd.read_csv("fear_greed_index.csv")

print("Trades Shape   :", trades.shape)
print("Sentiment Shape:", sentiment.shape)

### A2. Data Card — Rows, Columns, Missing Values, Duplicates

In [ ]:
def data_card(df, name):
    print(f"{'═'*55}")
    print(f"  DATASET : {name}")
    print(f"{'═'*55}")
    print(f"  Rows         : {df.shape[0]:,}")
    print(f"  Columns      : {df.shape[1]}")
    print(f"  Duplicate rows: {df.duplicated().sum():,}")
    print("  Columns list :", df.columns.tolist())
    print("\n  Missing values per column:")
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    miss_df = pd.DataFrame({'Count': miss, '%': miss_pct})
    cols_with_miss = miss_df[miss_df['Count'] > 0]
    print(cols_with_miss.to_string() if len(cols_with_miss) > 0 else "    None — clean!")
    print()

data_card(trades,    "Hyperliquid Trader Data (historical_data.csv)")
data_card(sentiment, "Bitcoin Fear/Greed Index (fear_greed_index.csv)")

### A3. Timestamp Conversion & Merge

In [ ]:
# Convert trader timestamps
trades.columns    = trades.columns.str.strip()
sentiment.columns = sentiment.columns.str.strip()

trades['date'] = pd.to_datetime(
    trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce').dt.date

# Convert Unix timestamp in sentiment (unit = seconds)
sentiment['timestamp'] = pd.to_datetime(sentiment['timestamp'], unit='s', errors='coerce')
sentiment['date']      = sentiment['timestamp'].dt.date

# Simplify: map Extreme Fear/Greed → Fear/Greed for binary analysis
sentiment['classification_simple'] = sentiment['classification'].map({
    'Fear': 'Fear', 'Extreme Fear': 'Fear',
    'Greed': 'Greed', 'Extreme Greed': 'Greed'
})

print("Trader date range   :", trades['date'].min(), "→", trades['date'].max())
print("Sentiment date range:", sentiment['date'].min(), "→", sentiment['date'].max())

In [ ]:
merged = pd.merge(
    trades,
    sentiment[['date', 'classification', 'classification_simple']],
    on='date',
    how='inner'
)

merged['Closed PnL'] = pd.to_numeric(merged['Closed PnL'], errors='coerce')
merged['Size USD']   = pd.to_numeric(merged['Size USD'],   errors='coerce')

print("Merged Shape  :", merged.shape)
print("Unique traders:", merged['Account'].nunique())
print("\nSentiment day distribution:")
print(merged.groupby('classification_simple')['date'].nunique().rename('unique_days'))
print(merged.head(3))

### A4. Key Metrics Engineering

In [ ]:
# Win flag
merged['win'] = merged['Closed PnL'] > 0

# Leverage proxy (no direct leverage column in dataset)
# Proxy = |Start Position| / Size USD — captures relative position sizing
merged['lev_proxy'] = (merged['Start Position'].abs() / merged['Size USD']
                       ).replace([np.inf, -np.inf], np.nan).clip(0, 100)

# Direction (Long / Short) from Direction column
long_dirs  = ['Buy', 'Open Long', 'Long > Short']
short_dirs = ['Sell', 'Open Short', 'Short > Long']
merged['direction'] = merged['Direction'].apply(
    lambda x: 'Long' if x in long_dirs else ('Short' if x in short_dirs else 'Other')
)

# Daily metrics
daily_metrics = merged.groupby(['date', 'classification_simple']).agg(
    daily_pnl    = ('Closed PnL', 'sum'),
    avg_trade_sz = ('Size USD',   'mean'),
    trade_count  = ('Account',    'count'),
    win_rate     = ('win',        'mean'),
).reset_index()

print("Daily metrics shape:", daily_metrics.shape)
print(daily_metrics.head())

---
## Part B — Analysis
### B1. Performance: PnL, Win Rate & Drawdown Proxy — Fear vs Greed

In [ ]:
perf_table = merged.groupby('classification_simple').agg(
    Trades         = ('Closed PnL', 'count'),
    Mean_PnL       = ('Closed PnL', 'mean'),
    Median_PnL     = ('Closed PnL', 'median'),
    Total_PnL      = ('Closed PnL', 'sum'),
    Std_PnL        = ('Closed PnL', 'std'),
    Win_Rate_pct   = ('win',        lambda x: round(x.mean() * 100, 2)),
    Drawdown_Proxy = ('Closed PnL', lambda x: round(x.quantile(0.05), 4))
).round(4)

print("═"*60)
print("  Performance Table: Fear vs Greed")
print("═"*60)
print(perf_table.T.to_string())
perf_table.to_csv('charts/performance_by_sentiment.csv')

In [ ]:
COLORS = {'Fear': '#e74c3c', 'Greed': '#27ae60'}
labels = ['Fear', 'Greed']
colors = [COLORS[l] for l in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chart 1 — PnL Distribution: Fear vs Greed Days', fontsize=13, fontweight='bold')

# Boxplot (clip extreme outliers for visibility)
clip_val  = merged['Closed PnL'].quantile(0.995)
plot_data = merged[merged['Closed PnL'].abs() < clip_val]
sns.boxplot(data=plot_data, x='classification_simple', y='Closed PnL',
            hue='classification_simple', palette=COLORS, legend=False,
            ax=axes[0], order=labels, width=0.5)
axes[0].set_title('PnL Spread (99.5th pct clipped)')
axes[0].set_xlabel('Market Sentiment')
axes[0].set_ylabel('Closed PnL (USD)')

# Mean PnL bar
mean_pnl = merged.groupby('classification_simple')['Closed PnL'].mean()[labels]
bars = axes[1].bar(labels, mean_pnl.values, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Mean PnL per Trade')
axes[1].set_xlabel('Market Sentiment')
axes[1].set_ylabel('Mean PnL (USD)')
for bar, val in zip(bars, mean_pnl.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'${val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart1_pnl_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Chart 2 — Win Rate & Drawdown Proxy: Fear vs Greed', fontsize=13, fontweight='bold')

wr = perf_table['Win_Rate_pct'][labels]
bars = axes[0].bar(labels, wr.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Win Rate (%)')
axes[0].set_ylabel('Win Rate (%)')
axes[0].set_ylim(0, 60)
for bar, val in zip(bars, wr.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')

dd = perf_table['Drawdown_Proxy'][labels]
bars2 = axes[1].bar(labels, dd.values, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Drawdown Proxy (5th Percentile PnL)')
axes[1].set_ylabel('PnL (USD)')
for bar, val in zip(bars2, dd.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val - 0.1,
                 f'${val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart2_winrate_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

### B2. Behavior Change by Sentiment — Trade Frequency, Leverage, Long/Short Bias

In [ ]:
# Long/Short ratio
ls_counts = (merged[merged['direction'].isin(['Long','Short'])]
             .groupby(['classification_simple','direction'])
             .size().unstack(fill_value=0))
ls_counts['L/S Ratio'] = (ls_counts['Long'] / ls_counts['Short']).round(3)
print("Long/Short Ratio by Sentiment:")
print(ls_counts)
ls_counts.to_csv('charts/long_short_ratio.csv')

# Leverage proxy by sentiment
print("\nLeverage Proxy by Sentiment:")
print(merged.groupby('classification_simple')['lev_proxy']
      .agg(['mean','median']).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chart 3 — Trader Behavior: Direction Bias & Daily Frequency by Sentiment',
             fontsize=13, fontweight='bold')

ls_plot = ls_counts[['Long','Short']].loc[labels]
ls_plot.plot(kind='bar', ax=axes[0], color=['#2980b9','#e67e22'], edgecolor='white', width=0.6)
axes[0].set_title('Long vs Short Trade Count')
axes[0].set_xticklabels(labels, rotation=0)
axes[0].set_ylabel('Number of Trades')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)
for i, (idx, row) in enumerate(ls_counts.loc[labels].iterrows()):
    axes[0].text(i, max(row['Long'], row['Short']) * 0.55,
                 f"L/S: {row['L/S Ratio']:.2f}x", ha='center',
                 color='white', fontweight='bold', fontsize=10)

daily_trades = merged.groupby(['date','classification_simple']).size().reset_index(name='count')
avg_daily    = daily_trades.groupby('classification_simple')['count'].mean()[labels]
bars = axes[1].bar(labels, avg_daily.values, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Average Daily Trade Count')
axes[1].set_ylabel('Avg Trades per Day')
for bar, val in zip(bars, avg_daily.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 2,
                 f'{val:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart3_behavior_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chart 4 — Leverage Proxy Distribution by Sentiment', fontsize=13, fontweight='bold')

lev_clip = merged['lev_proxy'].dropna()
lev_clip = lev_clip[lev_clip < lev_clip.quantile(0.99)]
axes[0].hist(lev_clip, bins=50, color='steelblue', edgecolor='white')
med = merged['lev_proxy'].median()
axes[0].axvline(med, color='red', linestyle='--', label=f'Median: {med:.1f}x')
axes[0].set_title('Overall Leverage Proxy Distribution')
axes[0].set_xlabel('Leverage Proxy')
axes[0].set_ylabel('Count')
axes[0].legend()

for label, color in [('Fear','#e74c3c'),('Greed','#27ae60')]:
    sub = merged[merged['classification_simple']==label]['lev_proxy'].dropna()
    sub = sub[sub < sub.quantile(0.99)]
    axes[1].hist(sub, bins=40, alpha=0.65, label=label, color=color, edgecolor='white')
axes[1].set_title('Leverage Proxy: Fear vs Greed Days')
axes[1].set_xlabel('Leverage Proxy')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('charts/chart4_leverage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### B3. Trader Segmentation — 3 Segments

In [ ]:
# ── Segment 1: Frequent vs Infrequent ────────────────────────────────────────
ta = merged.groupby('Account').size().reset_index(name='total_trades')
med_trades = ta['total_trades'].median()
ta['freq_segment'] = ta['total_trades'].apply(
    lambda x: 'Frequent' if x > med_trades else 'Infrequent')
merged = merged.merge(ta[['Account','freq_segment','total_trades']], on='Account', how='left')
print(f"Frequency threshold (median): {med_trades:.0f} trades")
print(ta['freq_segment'].value_counts())

# ── Segment 2: High vs Low Leverage ──────────────────────────────────────────
tl = merged.groupby('Account')['lev_proxy'].median().reset_index()
tl.columns = ['Account','med_lev']
lev_med = tl['med_lev'].median()
tl['lev_segment'] = tl['med_lev'].apply(
    lambda x: 'High Leverage' if x > lev_med else 'Low Leverage')
merged = merged.merge(tl[['Account','lev_segment']], on='Account', how='left')
print(f"\nLeverage threshold (median): {lev_med:.2f}x proxy")
print(tl['lev_segment'].value_counts())

# ── Segment 3: Consistent Winners vs Inconsistent ────────────────────────────
tc = merged.groupby('Account').agg(
    tr=('Closed PnL','count'), wr=('win','mean')).reset_index()
tc['consist_segment'] = 'Inconsistent'
tc.loc[(tc['wr'] > 0.55) & (tc['tr'] >= 10), 'consist_segment'] = 'Consistent Winner'
merged = merged.merge(tc[['Account','consist_segment']], on='Account', how='left')
print(f"\nConsistent Winner accounts (win rate >55%, ≥10 trades):",
      tc[tc['consist_segment']=='Consistent Winner']['Account'].nunique())
print(merged['consist_segment'].value_counts())

In [ ]:
seg_perf = merged.groupby('freq_segment').agg(
    Trades   = ('Closed PnL','count'),
    Avg_PnL  = ('Closed PnL','mean'),
    Win_Rate = ('win', lambda x: round(x.mean()*100,2))
).round(4)
print("Segment 1 — Frequent vs Infrequent:")
print(seg_perf)

lev_perf = merged.groupby('lev_segment').agg(
    Trades   = ('Closed PnL','count'),
    Avg_PnL  = ('Closed PnL','mean'),
    Win_Rate = ('win', lambda x: round(x.mean()*100,2))
).round(4)
print("\nSegment 2 — High vs Low Leverage:")
print(lev_perf)

cons_perf = merged.groupby(['consist_segment','classification_simple']).agg(
    Trades   = ('Closed PnL','count'),
    Avg_PnL  = ('Closed PnL','mean'),
    Win_Rate = ('win', lambda x: round(x.mean()*100,2))
).round(4)
print("\nSegment 3 — Consistent vs Inconsistent by Sentiment:")
print(cons_perf)
cons_perf.to_csv('charts/consistency_segment.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Chart 5 — Trader Segmentation: Win Rate Across 3 Segments',
             fontsize=13, fontweight='bold')

segments = [
    ('freq_segment',    ['Infrequent','Frequent'],           'Infrequent vs Frequent',  ['#3498db','#e67e22']),
    ('lev_segment',     ['Low Leverage','High Leverage'],    'Low vs High Leverage',    ['#9b59b6','#1abc9c']),
    ('consist_segment', ['Inconsistent','Consistent Winner'],'Inconsistent vs Consistent', ['#e74c3c','#2ecc71']),
]

for ax, (col, order, title, pal) in zip(axes, segments):
    wr_seg = merged.groupby(col)['win'].mean().mul(100).round(2).reindex(order)
    bars   = ax.bar(wr_seg.index, wr_seg.values, color=pal, edgecolor='white', width=0.55)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Win Rate (%)')
    ax.set_ylim(0, 90)
    ax.tick_params(axis='x', rotation=12)
    for bar, val in zip(bars, wr_seg.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('charts/chart5_all_segments.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part C — Actionable Strategy Recommendations

### Key Insights
1. **Fear days yield higher mean PnL ($59.28) vs Greed days ($45.56)** — Fear creates larger price dislocations which skilled traders exploit, but also higher volatility (std $1,038 vs $880).
2. **Long bias spikes during Fear (L/S ratio 1.44x) vs Greed (1.04x)** — Traders aggressively buy dips during Fear. This contrarian behavior pays off on average but exposes them to deeper drawdowns (5th pct: -$3.01 Fear vs -$6.04 Greed).
3. **Consistent Winners (80.1% win rate) far outperform Inconsistent traders (39.1%)** — Only 2 accounts qualify, suggesting most traders lack repeatable edge. Their advantage holds across both Fear and Greed days.
4. **Frequent traders (41.5% win rate) marginally outperform Infrequent (37.9%)** — Experience helps, but the gap is small, suggesting frequency alone doesn't build edge.
5. **High leverage traders win more often (42.8%) but with higher variance** — Size amplifies both wins and losses, making leverage a double-edged sword especially on Fear days.

---

### Strategy 1 — Fear Day Playbook
> **Rule:** During Fear days, maintain or slightly increase trade frequency (data shows 477 avg daily trades vs 430 on Greed days). However, **cap leverage proxy below your personal median** and reduce position size by 20–30%. Bias towards Long entries (L/S 1.44x is data-confirmed), but set tighter stop-losses given higher PnL std dev.
>
> **For Inconsistent traders specifically:** Sit out Fear days entirely, or paper-trade only. The $59.28 mean PnL is driven by a small number of high-conviction traders — Inconsistent traders contribute disproportionately to the -$3.01 5th-percentile drawdown.

### Strategy 2 — Greed Day Playbook
> **Rule:** During Greed days, normalize your long/short ratio toward 1:1 (the data shows Greed L/S at 1.04x — near balanced). **Frequent traders should cap daily trades at 1.2x their personal average** — the lower mean PnL ($45.56) on Greed days means overtrading eats into edge. Use the higher daily trade volume environment to scale out of winning positions rather than opening new ones.
>
> **For High Leverage traders specifically:** Reduce leverage on Greed days. The deeper Greed drawdown proxy (-$6.04) combined with the crowded long positioning means leverage amplifies loss when reversals happen.


---
## Save All Outputs

In [ ]:
# Save all summary tables
perf_table.to_csv('charts/performance_by_sentiment.csv')
ls_counts.to_csv('charts/long_short_ratio.csv')
cons_perf.to_csv('charts/consistency_segment.csv')
daily_metrics.to_csv('charts/daily_metrics.csv', index=False)

print("Saved files:")
print([f for f in os.listdir('charts')])